# 薪資預測 API 教學：實作 `/predict` 預測端點

> 對象：修習多元線性迴歸 / 模型部署課程的學生
> 對應程式碼：`backend/test/app.py` 第 86~110 行的 `predict_api` 函數
>
> 這份 notebook 會帶你**一步步拆解**預測流程，並在最後**自己補完**一個可用的預測函數。

## 學習目標

- [ ] 看懂 FastAPI 端點的基本結構（裝飾器、請求模型、回應模型）
- [ ] 能說明預測前需要做哪 4 個預處理步驟
- [ ] 能獨立完成：學歷編碼 → 城市編碼 → 組特徵 → 標準化 → 預測
- [ ] 能處理「未知學歷」的錯誤（回 400）
- [ ] 能用 `TestClient` 呼叫真實 API 端點並檢查回應

## 背景：整個預測流程

訓練階段已經把「模型」和「預處理器」一起存進 `salary_model.joblib`：

```
模型檔 (salary_model.joblib)
├── model   : 訓練好的線性迴歸模型
├── oe      : OrdinalEncoder   (學歷 → 0/1/2)
├── ohe     : OneHotEncoder    (城市 → 城市A/B/C 三個 0/1 欄位)
└── scaler  : StandardScaler   (標準化)
```

預測時，`/predict` 收到使用者的資料後，要做的是**和訓練時一模一樣的步驟**：

```
年資(years_experience) ─┐
學歷(education_level) ──┼─> 特徵向量 ─> scaler.transform ─> model.predict ─> 薪資
城市(city) ────────────┘
     (oe編碼)      (ohe編碼)
```

接下來就照這個流程動手做。

In [1]:
# ============================================
# 0. 載入套件與模型
# ============================================

import os, sys
import joblib
import pandas as pd
import numpy as np
from pprint import pprint
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

current_dir = os.getcwd()          # notebook 所在資料夾 = backend/test
model_path  = os.path.join(current_dir, "salary_model.joblib")

model_data = joblib.load(model_path)

print("model_data 內含的 key：")
pprint(list(model_data.keys()))

model_data 內含的 key：
['model',
 'oe',
 'ohe',
 'scaler',
 'r2',
 'coef',
 'intercept',
 'feature_names',
 'feature_coefs',
 'model_type',
 'alpha',
 'train_time',
 'test_size',
 'random_state']


In [2]:
# ============================================
# 認識 MODEL_STATE（模擬 app.py 裡的全域狀態）
# ============================================

MODEL_STATE = {
    "model": model_data["model"],
    "oe": model_data.get("oe"),
    "le": model_data.get("le"),
    "ohe": model_data["ohe"],
    "scaler": model_data["scaler"],
    "r2": model_data.get("r2"),
    "feature_names": model_data["feature_names"],
    "feature_coefs": model_data.get("feature_coefs", {}),
    "model_type": model_data.get("model_type"),
}

print("模型       :", MODEL_STATE["model"])
print("模型類型   :", MODEL_STATE["model_type"])
print("R² Score   :", MODEL_STATE["r2"])
print("特徵欄位順序:", MODEL_STATE["feature_names"])
print()
print("預處理器：")
print("  oe     (學歷 OrdinalEncoder):", MODEL_STATE["oe"])
print("  ohe    (城市 OneHotEncoder) :", MODEL_STATE["ohe"])
print("  scaler (StandardScaler)    :", MODEL_STATE["scaler"])

模型       : LinearRegression()
模型類型   : LinearRegression
R² Score   : 0.8462535226367867
特徵欄位順序: ['YearsExperience', 'EducationLevel', 'City_城市A', 'City_城市B', 'City_城市C']

預處理器：
  oe     (學歷 OrdinalEncoder): OrdinalEncoder(categories=[['高中以下', '大學', '碩士以上']])
  ohe    (城市 OneHotEncoder) : OneHotEncoder(handle_unknown='ignore', sparse_output=False)
  scaler (StandardScaler)    : StandardScaler()


---
## Part 1：認識 API 端點的結構

`app.py` 第 86~110 行的端點長這樣（先看骨架，中間邏輯稍後自己補）：

```python
@api_app.post("/predict", response_model=SalaryOutput)
def predict_api(payload: SalaryInput):
    # 1. 取出 MODEL_STATE 裡的預處理器與模型
    # 2. 學歷用 oe 編碼（未知學歷回 400）
    # 3. 城市用 ohe 編碼
    # 4. 組特徵 + 標準化
    # 5. model.predict 預測，回傳月薪與年薪
    try:
        ...
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"預測失敗:{str(e)}")
```

三個重點：
1. **`@api_app.post("/predict")`**：告訴 FastAPI「收到 POST /predict 就呼叫這個函數」
2. **`payload: SalaryInput`**：FastAPI 自動把 JSON body 轉成這個 Pydantic 物件（等同型別檢查）
3. **`response_model=SalaryOutput`**：回傳前 FastAPI 會驗證格式，確保欄位正確

In [3]:
# ============================================
# 定義請求與回應的 Pydantic 模型（與 app.py 相同）
# ============================================

from pydantic import BaseModel, Field

class SalaryInput(BaseModel):
    years_experience: float = Field(..., description="工作年資 (年)", ge=0.0, le=50.0)
    education_level: str = Field(..., description="學歷 (高中以下、大學、碩士以上)")
    city: str = Field(..., description="工作城市 (城市A、城市B、城市C)")

class SalaryOutput(BaseModel):
    predicted_salary: float = Field(..., description="預測月薪 (k / 千元)")
    estimated_annual_salary: float = Field(..., description="估計年薪 (k / 千元，以 14 個月估算)")

print("SalaryInput 的 JSON Schema（前端要照這個格式送資料）：")
print(SalaryInput.model_json_schema())

SalaryInput 的 JSON Schema（前端要照這個格式送資料）：
{'properties': {'years_experience': {'description': '工作年資 (年)', 'maximum': 50.0, 'minimum': 0.0, 'title': 'Years Experience', 'type': 'number'}, 'education_level': {'description': '學歷 (高中以下、大學、碩士以上)', 'title': 'Education Level', 'type': 'string'}, 'city': {'description': '工作城市 (城市A、城市B、城市C)', 'title': 'City', 'type': 'string'}}, 'required': ['years_experience', 'education_level', 'city'], 'title': 'SalaryInput', 'type': 'object'}


---
## Part 2：一步步拆解預處理流程

> 下面 5 個步驟，就是 `predict_api` 裡最重要的內容。
> 我會先把步驟 1 完整寫給你看，步驟 2~5 留空格給你填。

### 步驟 1：學歷 → Ordinal 編碼

`OrdinalEncoder` 把「高中以下、大學、碩士以上」依序轉成 0、1、2。
注意：**轉換的輸入必須是 2 維**，所以用 `pd.DataFrame([[學歷]], columns=[...])` 包起來。

In [4]:
# 步驟 1（完整示範）
oe = MODEL_STATE["oe"]

edu = "碩士以上"
edu_encoded = int(oe.transform(
    pd.DataFrame([[edu]], columns=["EducationLevel"])
)[0][0])
display(int(oe.transform(pd.DataFrame([[edu]], columns=["EducationLevel"]))[0][0]))

print(f"學歷「{edu}」 -> 編碼 {edu_encoded}")

2

學歷「碩士以上」 -> 編碼 2


In [5]:
# 練習：試試不同學歷，觀察合法與不合法的結果
for edu in ["高中以下", "大學", "碩士以上", "博士"]:
    try:
        v = int(oe.transform(pd.DataFrame([[edu]], columns=["EducationLevel"]))[0][0])
        print(f"{edu:6s} -> {v}")
    except ValueError as e:
        print(f"{edu:6s} -> ValueError（未知類別）")

print()
print("可接受的學歷:", list(oe.categories_[0]))

高中以下   -> 0
大學     -> 1
碩士以上   -> 2
博士     -> ValueError（未知類別）

可接受的學歷: ['高中以下', '大學', '碩士以上']


### 步驟 2：城市 → OneHot 編碼

城市有 3 種可能（城市A / 城市B / 城市C），用 OneHot 轉成 3 個 0/1 欄位，避免「城市A=0、城市B=1、城市C=2」這種錯誤的數字大小關係。

In [6]:
# 步驟 2（填空）
ohe = MODEL_STATE["ohe"]
city = "城市B"

# TODO ①：把 city 用 ohe.transform 轉成 OneHot 向量
#   提示：ohe.transform(pd.DataFrame([[city]], columns=["City"]))
city_vector = ohe.transform(pd.DataFrame([[city]],columns=["City"]))  # ← 你的程式碼
print(city_vector)

# 取得 OneHot 的欄位名稱（city_cols 這行已寫好，不用改）
city_cols = ohe.get_feature_names_out(["City"])

print("城市欄位:", list(city_cols))
print("OneHot 向量:", city_vector)


[[0. 1. 0.]]
城市欄位: ['City_城市A', 'City_城市B', 'City_城市C']
OneHot 向量: [[0. 1. 0.]]


### 步驟 3：組合成特徵 DataFrame

模型的輸入特徵順序是：

```
['YearsExperience', 'EducationLevel', 'City_城市A', 'City_城市B', 'City_城市C']
```

所以要把「年資」、「學歷編碼」和「城市 OneHot」這 3 段拼成一列。

In [7]:
# 步驟 3（填空）
years_experience = 5.3

# TODO ②：把 [年資, 學歷編碼] 加上城市 OneHot 串成一個特徵列
feature_row = [years_experience,edu_encoded] + city_vector[0].tolist()  # ← 你的程式碼（提示：list 相加）

feature_names = ["YearsExperience", "EducationLevel"] + list(city_cols)
features = pd.DataFrame([feature_row], columns=feature_names)
features

,YearsExperience,EducationLevel,City_城市A,City_城市B,City_城市C
0,5.3,2,0.0,1.0,0.0


### 步驟 4：標準化（StandardScaler）

訓練時用 `scaler.fit_transform(X_train)` 擬合過，預測時**只能**用 `scaler.transform`（不能重新 fit，否則統計值會和訓練不一致）。

In [8]:
# 步驟 4（填空）
scaler = MODEL_STATE["scaler"]

# TODO ③：把 features 標準化
X_scaled = scaler.transform(features)  # ← 你的程式碼（提示：scaler.transform）

pd.DataFrame(X_scaled, columns=features.columns)

,YearsExperience,EducationLevel,City_城市A,City_城市B,City_城市C
0,-0.028279,1.021466,-1.154701,3.605551,-0.745356


### 步驟 5：預測月薪與年薪

`model.predict` 回傳陣列，取 `[0]` 得到單一預測值（單位為 k / 千元）。
年薪假設 14 個月，所以 `月薪 * 14`。

In [9]:
# 步驟 5（填空）
model = MODEL_STATE["model"]

# TODO ④：用模型預測月薪，並算出估計年薪
predicted_salary = model.predict(X_scaled)[0]       # ← 你的程式碼（提示：model.predict(X_scaled)[0]）
estimated_annual_salary = predicted_salary * 14 # ← 你的程式碼（提示：月薪 * 14）

print(f"預測月薪 : {predicted_salary:.2f} k")
print(f"估計年薪 : {estimated_annual_salary:.2f} k")

預測月薪 : 52.35 k
估計年薪 : 732.91 k


---
## Part 3：把流程串成一個 `predict_api` 函數

把步驟 1~5 整合成一個完整函數（跟 `app.py` 第 86 行起的那支端點邏輯一樣，只是先不接 FastAPI）。其中 (1) 已寫好作為範例，請完成 (2)~(6)。

In [10]:
# ============================================
# 完成 predict_api（填空）
# ============================================

from fastapi import HTTPException

def predict_api(years_experience: float, education_level: str, city: str) -> dict:
    oe     = MODEL_STATE["oe"]
    ohe    = MODEL_STATE["ohe"]
    scaler = MODEL_STATE["scaler"]
    model  = MODEL_STATE["model"]

    # (1) 學歷編碼 + 錯誤處理（範例，已寫好）
    try:
        edu_encoded = int(oe.transform(
            pd.DataFrame([[education_level]], columns=["EducationLevel"])
        )[0][0])
    except ValueError:
        valid_cats = list(oe.categories_[0])
        raise HTTPException(
            status_code=400,
            detail=f"未知的學歷:{education_level}. 可接受的值為:{valid_cats}",
        )

    # (2) TODO ⑤：城市 OneHot 編碼
    city_vector = ohe.transform(pd.DataFrame([[city]], columns=["City"]))
    city_cols   = ohe.get_feature_names_out(["City"])

    # (3) TODO ⑥：組合成特徵 DataFrame
    feature_row = [years_experience, edu_encoded] + list(city_vector[0])
    features    = pd.DataFrame([feature_row], columns=["YearsExperience", "EducationLevel"] + list(city_cols))

    # (4) TODO ⑦：標準化
    X_scaled = scaler.transform(features)

    # (5) TODO ⑧：預測月薪
    predicted_salary = float(model.predict(X_scaled)[0])

    # (6) TODO ⑨：回傳與 SalaryOutput 欄位一致的 dict
    return {
        "predicted_salary": predicted_salary,
        "estimated_annual_salary": predicted_salary * 14,
    }

#### 手動測試你的 `predict_api`

In [11]:
# 測試 1：正常輸入
result = predict_api(
    years_experience=5.3,
    education_level="碩士以上",
    city="城市A",
)
print(result)

# 對照組：用課堂上教的公式手算 (5 個特徵的線性組合)
# predicted = intercept + Σ(coef * 標準化後特徵)

{'predicted_salary': 67.19386521799288, 'estimated_annual_salary': 940.7141130519003}


In [12]:
# 測試 2：未知學歷 -> 應該回 400
try:
    predict_api(5.3, "博士", "城市A")
except HTTPException as e:
    print("HTTPException status:", e.status_code)
    print("detail:", e.detail)

HTTPException status: 400
detail: 未知的學歷:博士. 可接受的值為:['高中以下', '大學', '碩士以上']


---
## Part 4：把函數接上 FastAPI，用 `TestClient` 測試

FastAPI 端點只是「接收請求 → 呼叫 `predict_api` → 回傳」。我們在 notebook 裡建一個**迷你 API** 來模擬 `app.py` 的 `/predict`，並用 `TestClient` 送真實 HTTP 請求（不需要真的啟動伺服器）。

In [13]:
# 建立迷你 FastAPI 應用（模擬 app.py 的端點）
from fastapi import FastAPI
from fastapi.testclient import TestClient

mini_app = FastAPI()

@mini_app.post("/predict", response_model=SalaryOutput)
def predict_endpoint(payload: SalaryInput):
    # 端點只負責接資料，真正的邏輯交給 predict_api
    return predict_api(
        years_experience=payload.years_experience,
        education_level=payload.education_level,
        city=payload.city,
    )

client = TestClient(mini_app)

# 送一個正常的預測請求
response = client.post("/predict", json={
    "years_experience": 5.3,
    "education_level": "碩士以上",
    "city": "城市A",
})
print("HTTP 狀態碼:", response.status_code)
print("回應內容:", response.json())

HTTP 狀態碼: 200
回應內容: {'predicted_salary': 67.19386521799288, 'estimated_annual_salary': 940.7141130519003}


/Users/roberthsu2003/Documents/GitHub/2026_07_03_python_ai_tvdi/.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [14]:
# 錯誤測試：看看不同錯誤各回什麼狀態碼
resp1 = client.post("/predict", json={
    "years_experience": 5.3,
    "education_level": "博士",   # 未知學歷
    "city": "城市A",
})
print("未知學歷 ->", resp1.status_code, resp1.json())
print()

resp2 = client.post("/predict", json={
    "years_experience": -3,       # 違反 ge=0 的限制
    "education_level": "大學",
    "city": "城市A",
})
print("年資為負 ->", resp2.status_code, resp2.json())
print()

resp3 = client.post("/predict", json={
    "years_experience": 5.3,
    "education_level": "大學",
    "city": "台北",              # 未知城市
})
print("未知城市 ->", resp3.status_code, resp3.json())

未知學歷 -> 400 {'detail': "未知的學歷:博士. 可接受的值為:['高中以下', '大學', '碩士以上']"}

年資為負 -> 422 {'detail': [{'type': 'greater_than_equal', 'loc': ['body', 'years_experience'], 'msg': 'Input should be greater than or equal to 0', 'input': -3, 'ctx': {'ge': 0.0}}]}

未知城市 -> 200 {'predicted_salary': 47.01569812545799, 'estimated_annual_salary': 658.2197737564118}


### 觀察與思考

1. 「未知學歷」走的是哪一行程式碼？回傳的 `detail` 內容長什麼樣？
2. 「年資為負」不是程式自己檢查的，而是哪個地方擋下來的？→ 答案是 `Field(..., ge=0.0)` 的驗證規則，FastAPI 會自動回 **422**。
3. 「未知城市」目前的結果是什麼？想想看，`ohe` 訓練時設了 `handle_unknown='ignore'`，這代表什麼？（提示：對應的 OneHot 欄位全為 0）

---
## 作業 / 挑戰題

1. **完成 `predict_api`**：把 Part 3 的 TODO ⑤~⑨ 全部補完，並通過「測試 1 / 測試 2」。
2. **驗證年資影響**：固定學歷與城市，用年資 1、5、10 各呼叫一次 `predict_api`，觀察薪水趨勢，說明是否符合迴歸直覺。
3. **觀察 `app.py` 的陷阱**：`app.py` 第 109 行有 `except Exception` 包住整段邏輯。當「未知學歷」的 `HTTPException(400)` 在內層被丟出時，會被外層的 `except Exception` 接住，變成回 **500**「預測失敗:400: ...」。請回答：
   - 為什麼會這樣？（提示：`HTTPException` 也是 `Exception` 的一種）
   - 要怎麼修正，讓「未知學歷」正確回 **400**？（提示：讓內層例外不要被外層 catch，例如直接 `return JSONResponse(...)` 或把 `HTTPException` 擋在外面）
4. **加分題**：在 `predict_api` 裡加上「未知城市」的檢查（模仿學歷的做法），回 400。

---
## 教師解答區（發給學生前請刪除本 cell 以下內容）

### 步驟 2（TODO ①）
```python
city_vector = ohe.transform(pd.DataFrame([[city]], columns=["City"]))
```

### 步驟 3（TODO ②）
```python
feature_row = [years_experience, edu_encoded] + list(city_vector[0])
```

### 步驟 4（TODO ③）
```python
X_scaled = scaler.transform(features)
```

### 步驟 5（TODO ④）
```python
predicted_salary = model.predict(X_scaled)[0]
estimated_annual_salary = predicted_salary * 14
```

### predict_api（TODO ⑤~⑨）
```python
from fastapi import HTTPException, JSONResponse

def predict_api(years_experience: float, education_level: str, city: str) -> dict:
    oe     = MODEL_STATE["oe"]
    ohe    = MODEL_STATE["ohe"]
    scaler = MODEL_STATE["scaler"]
    model  = MODEL_STATE["model"]

    try:
        edu_encoded = int(oe.transform(
            pd.DataFrame([[education_level]], columns=["EducationLevel"])
        )[0][0])
    except ValueError:
        valid_cats = list(oe.categories_[0])
        return JSONResponse(
            status_code=400,
            content={"detail": f"未知的學歷:{education_level}. 可接受的值為:{valid_cats}"},
        )

    city_vector = ohe.transform(pd.DataFrame([[city]], columns=["City"]))
    city_cols   = ohe.get_feature_names_out(["City"])

    feature_row = [years_experience, edu_encoded] + list(city_vector[0])
    features    = pd.DataFrame(
        [feature_row],
        columns=["YearsExperience", "EducationLevel"] + list(city_cols),
    )

    X_scaled = scaler.transform(features)
    predicted_salary = float(model.predict(X_scaled)[0])

    return {
        "predicted_salary": predicted_salary,
        "estimated_annual_salary": predicted_salary * 14,
    }
```

> 修正 `app.py` 的重點：**不要讓 400 的 `HTTPException` 被外層 `except Exception` 吃掉**。
> 簡單做法是用 `JSONResponse` 直接回傳（如上），或把學歷檢查移到 try/except 之外。
> 另外提醒：`oe` 可能是 `None`（舊模型檔），若要更嚴謹可先檢查 `oe is not None`。